# Visualization

Every earlier notebook treated a polar signal as a plain array and left plotting to
whatever `matplotlib` call the reader wrote by hand. `pypft.viz` turns that into a
small, consistent API: `pypft.plot_signal` (and its `pypft.BaseSignal.plot`
shortcut) renders one signal as a `(magnitude, phase)` pair of images, the same
way for every `pypft.Domain`; `pypft.render_cartesian` interpolates a
`POLAR`-domain signal back onto an ordinary Cartesian grid, purely for display;
and `pypft.forward_pft_traced`/`pypft.inverse_pft_traced` run the same pipeline
as `pypft.forward_pft`/`pypft.inverse_pft` while recording every domain -- and,
optionally, every figure, including a `PFTTrace.save` to disk -- along the way.

In [ ]:
%matplotlib inline

import cv2
import matplotlib.pyplot as plt
import numpy as np

import pypft

## A test image, already sampled onto the transform's own grid

`tests/samples/hedge_maze.tif` is a hedge-maze icon -- concentric rings with
angular gaps, the kind of image polar sampling suits naturally -- already
sampled by `scripts/make_test_image.py` onto
`pypft.PolarGrid(n_radial=576, n_angular=39, R=121.6)` via
`pypft.sample_cartesian`, so loading it here is a plain `cv2.imread`; no
`sample_cartesian` call is needed, unlike an ordinary square photo (which
would still need one). `n_radial=576`/`n_angular=39` were chosen so
`pypft.check_adequacy` raises no warning at this grid, while keeping the
per-harmonic kernel stack `forward_pft`/`inverse_pft` build under ~200MB
(see `scripts/make_test_image.py`'s own docstring for the cost that scales
with). See `THIRD-PARTY-NOTICES.md` for this image's license/attribution.

In [ ]:
image = cv2.imread("../tests/samples/hedge_maze.tif", cv2.IMREAD_GRAYSCALE)
values = image.astype(np.float64)  # (n_radial, n_angular), already polar-sampled

size = 256  # the square raster this was sampled from, before polar sampling
grid = pypft.PolarGrid(n_radial=576, n_angular=39, R=0.95 * size / 2)
signal = pypft.SpacePolarSignal(values=values, grid=grid)

## Plotting a signal: `plot_signal`/`BaseSignal.plot`

`pypft.plot_signal`/`signal.plot()` render a signal as a `(magnitude, phase)` pair
of images, for every domain -- PyPFT assumes `Domain.SPACE_POLAR` is
complex-valued too, the same as the frequency-domain members, since a full
forward-then-inverse round trip can leave even a `SPACE_POLAR` signal with a
non-trivial phase (see the round trip further below). Both return the `Axes`
used, as a tuple, so a caller can keep annotating the same plot afterward. The
signal loaded above is real-valued, so its phase panel below is uniformly zero
-- it only becomes interesting once a transform actually introduces one.
`Domain.SPACE_POLAR`'s own magnitude is drawn in grayscale rather than
`matplotlib`'s default colormap -- it is literally a photographic image, unlike
every other domain's more abstract magnitude.

In [ ]:
space_polar_magnitude_ax, space_polar_phase_ax = signal.plot()
plt.show()

### `Domain.SPACE_HARMONIC` is complex-valued too

`Domain.SPACE_HARMONIC` -- reached via `signal.to_harmonics()`, the first step
of the forward chain -- is an angular DFT's own coefficients, generically
complex even when the space-domain signal being transformed is real (a DFT of
real input is only symmetric, not real, in general), so it renders as a
(magnitude, phase) pair too, the same as every other domain. Unlike the
uniformly-zero phase above, `signal`'s own angular asymmetry -- the maze's
gaps aren't evenly spaced -- gives this one a genuinely non-trivial phase
already, without waiting for a full transform:

In [ ]:
harmonic_signal = signal.to_harmonics()
harmonic_magnitude_ax, harmonic_phase_ax = harmonic_signal.plot()
plt.show()

## The frequency domain: magnitude and phase

`Domain.FREQUENCY_HARMONIC`/`Domain.FREQUENCY_POLAR` render the same way
`Domain.SPACE_POLAR`/`Domain.SPACE_HARMONIC` did above: a gamma-enhanced
magnitude (`matplotlib.colors.PowerNorm`, not a hand-rolled `** gamma`) and a
phase map, since a frequency-domain magnitude typically spans a much larger
dynamic range than a plain real-valued image's. `harmonic_signal.to_frequency()`
reaches `Domain.FREQUENCY_HARMONIC` directly:

In [ ]:
frequency_harmonic_signal = harmonic_signal.to_frequency()
frequency_harmonic_magnitude_ax, frequency_harmonic_phase_ax = frequency_harmonic_signal.plot()
plt.show()

Converting `signal` all the way to `Domain.FREQUENCY_POLAR` produces its own
non-trivial phase too:

In [ ]:
frequency_signal = signal.to(pypft.Domain.FREQUENCY_POLAR)
magnitude_ax, phase_ax = frequency_signal.plot()
plt.show()

## Cartesian rendering, for display only

`pypft.render_cartesian` interpolates a `POLAR`-domain signal's own non-uniform
sample points (`pypft.PolarGrid.r`/`.theta`) onto an ordinary Cartesian pixel grid
via `scipy.interpolate.griddata` -- the two `HARMONIC` domains have no physical
angle axis to interpolate against, so only `SPACE_POLAR`/`FREQUENCY_POLAR` signals
are accepted. This is an approximation for display purposes only: its output must
never be fed back into `pypft.forward_pft`/`pypft.inverse_pft`, unlike
`pypft.sample_cartesian`'s exact, order-dependent sampling. `signal` renders in
grayscale below, the same as it did above -- `Domain.SPACE_POLAR`'s own colormap,
regardless of which function draws it.

In [ ]:
render_ax = pypft.render_cartesian(signal=signal, height=256, width=256)
plt.show()

## Tracing the whole pipeline

`pypft.forward_pft_traced`/`pypft.inverse_pft_traced` walk the same
`pypft.domains.BaseSignal` chain `pypft.forward_pft`/`pypft.inverse_pft` are built
on, so their `values` match exactly -- but they also return every intermediate
`pypft.BaseSignal` along the way, and can build figures for them:
`visualize_steps=True` renders one figure per domain (four, for the full chain),
and `visualize_pipeline=True` renders one holistic mosaic of all of them. Both
keywords combine freely, and every combination always returns the same `PFTTrace`
type. The forward trace below sets `visualize_steps=True`, so it returns four step
figures, each rendered exactly the way `plot_signal` renders that domain on its
own -- including `SPACE_POLAR`'s own grayscale magnitude.

In [ ]:
trace = pypft.forward_pft_traced(f=values, grid=grid, visualize_steps=True)

expected = pypft.forward_pft(f=values, grid=grid)
float(np.abs(trace.values - expected).max()), len(trace.figures)

### The way back: a full round trip

`pypft.inverse_pft_traced` is the exact mirror of `pypft.forward_pft_traced`:
applying it to `trace.values` (the frequency-domain result above) walks back
through the same four domains in reverse, ending at a reconstructed
`Domain.SPACE_POLAR` signal that should match `values` almost exactly, since
the round trip is exact up to numerical error. `PFTTrace.signals` keeps every
`pypft.BaseSignal` a trace visited, so `pypft.render_cartesian` can still be
called directly on either endpoint -- `inverse_trace.signals[0]` and
`inverse_trace.signals[-1]` -- which is handy when composing a custom
multi-signal comparison figure like the one below, rather than reusing one of
the trace's own step figures.

In [ ]:
inverse_trace = pypft.inverse_pft_traced(F=trace.values, grid=grid, visualize_steps=True)

float(np.abs(inverse_trace.values - values).max())

`inverse_trace.signals[0]` and `inverse_trace.signals[-1]` are its
`FREQUENCY_POLAR` and `SPACE_POLAR` endpoints. Rendering each directly with
`pypft.render_cartesian`, side by side with the very first Cartesian render of
the original signal, for comparison:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(
    render_ax.images[0].get_array(), origin="lower", cmap=render_ax.images[0].get_cmap()
)
axes[0].set_title("original (render_cartesian)")
pypft.render_cartesian(
    signal=inverse_trace.signals[0], height=size, width=size, ax=axes[1]
)
axes[1].set_title("frequency domain (render_cartesian)")
pypft.render_cartesian(
    signal=inverse_trace.signals[-1], height=size, width=size, ax=axes[2]
)
axes[2].set_title("reconstructed (render_cartesian)")
fig.tight_layout()
plt.show()

`PFTTrace.save` writes every figure in a trace to disk, enumerated and named
by domain -- the one place this module ever touches the filesystem, and only
when a caller asks for it explicitly:

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as directory:
    paths = inverse_trace.save(directory=Path(directory))
    names = [path.name for path in paths]

names

## The holistic mosaic

`visualize_pipeline=True` instead renders one figure holding every domain's own
(magnitude, phase) panel pair side by side. Both keywords can be set together;
each independently controls whether its own kind of figure gets built. Either
way, `forward_pft_traced`/`inverse_pft_traced` always return the same
`PFTTrace` type -- a return type that depended on which keywords were set
would be a `pyright` defect, which is exactly why tracing is a separate entry
point instead of a `visualize=` flag on `forward_pft`/`inverse_pft`
themselves.

In [ ]:
trace.close()  # done with the four step figures above
inverse_trace.close()  # ditto

mosaic_trace = pypft.forward_pft_traced(f=values, grid=grid, visualize_pipeline=True)
plt.show()

## Closing figures

`matplotlib` warns once more than 20 figures are open at once -- harmless
ordinarily, but this project's own test suite (`pytest.ini_options:
filterwarnings = ["error"]`) turns every warning into a failure, and a
long-running interactive session can hit the same limit just by tracing a few
signals without cleaning up. `PFTTrace.close()` closes every figure a trace
created, restoring `matplotlib.pyplot.get_fignums()` to its length beforehand; a
bare `plot_signal`/`render_cartesian` call is closed the same way, directly with
`matplotlib.pyplot.close`.

In [ ]:
mosaic_trace.close()
plt.close(render_ax.figure)
plt.close(space_polar_magnitude_ax.figure)
plt.close(harmonic_magnitude_ax.figure)
plt.close(frequency_harmonic_magnitude_ax.figure)
plt.close(magnitude_ax.figure)
plt.close(fig)  # the render_cartesian comparison mosaic above

len(plt.get_fignums())  # every figure this notebook created is now closed

## Where to go next

`pypft.viz` adds nothing numerical -- `plot_signal`/`render_cartesian` only ever
read a signal's own `values`, and `forward_pft_traced`/`inverse_pft_traced` compute
exactly what `forward_pft`/`inverse_pft` already compute, one verified domain step
at a time. What it adds is a consistent way to *see* a signal at any point along
the chain, and to inspect a whole transform's pipeline at once. PyPFT's remaining
phase builds a command-line interface on top of these same figures.